# LAB 08 — Tokenization (mục 10, 12.1)

Tokenizer: `distilbert-base-uncased` (WordPiece, uncased, vocab 30,522). Tokenizer lấy **cùng checkpoint** với encoder dùng ở `fine_tuning.ipynb`.

Mọi con số bên dưới là **output của lần chạy thật** — không đoán trước tokenization. Các câu nhận xét kết quả được in bằng code (f-string) từ chính output để không lệch số.

In [1]:
import pandas as pd  # Windows: import pandas truoc torch
import torch
from transformers import AutoTokenizer
import re, json, numpy as np

MODEL = "distilbert-base-uncased"
tok = AutoTokenizer.from_pretrained(MODEL)
print("tokenizer class :", type(tok).__name__)
print("vocab size      :", tok.vocab_size)
print("special tokens  :", tok.special_tokens_map)
print("special token ids:", {t: tok.convert_tokens_to_ids(t) for t in tok.all_special_tokens})
print("max model length :", tok.model_max_length)

C:\Users\Admin\AppData\Roaming\Python\Python314\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


tokenizer class : BertTokenizer
vocab size      : 30522
special tokens  : {'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}
special token ids: {'[UNK]': 100, '[SEP]': 102, '[PAD]': 0, '[CLS]': 101, '[MASK]': 103}
max model length : 512


## 1. Calculation Exercise 2 — `"The cats are playing."`

In [2]:
text = "The cats are playing."
tokens = tok(text, padding=True, truncation=True, return_tensors="pt")

print("tokens.keys() =", list(tokens.keys()))
ids = tokens["input_ids"][0].tolist()
mask = tokens["attention_mask"][0].tolist()
toks = tok.convert_ids_to_tokens(ids)
print("\ntext           :", text)
print("token sequence :", toks)
print("token IDs      :", ids)
print("attention_mask :", mask)

specials = [(i, t) for i, t in enumerate(toks) if t in tok.all_special_tokens]
words = re.findall(r"\w+", text)            # 'words' theo tu tach boi khoang trang/dau cau
content = [t for t in toks if t not in tok.all_special_tokens]
subword_pieces = [t for t in content if t.startswith("##")]
print("\nspecial tokens (vi tri, token):", specials)
print(f"so words                         : {len(words)}  {words}")
print(f"so tokens (gom special)          : {len(toks)}")
print(f"so tokens (khong gom special)    : {len(content)}  {content}")
print(f"so subword tokens (bat dau '##') : {len(subword_pieces)}  {subword_pieces}")
print("\ndecode:", tok.decode(ids))
# nhan xet sinh tu output
split_words = [w for w in words if len(tok.tokenize(w)) > 1]
print("\n[Nhan xet tu output]")
print(f"- {len(words)} words -> {len(content)} content tokens (+{len(specials)} special = {len(toks)}).")
print("- dau cau '.' tro thanh 1 token rieng." if "." in content else "- dau cau khong tach rieng.")
print(f"- tu bi tach thanh >1 piece: {split_words if split_words else 'khong co (moi word = 1 token)'}")
print("- uncased: 'The' duoc ha chu thuong thanh", repr(content[0]))

tokens.keys() = ['input_ids', 'token_type_ids', 'attention_mask']

text           : The cats are playing.
token sequence : ['[CLS]', 'the', 'cats', 'are', 'playing', '.', '[SEP]']
token IDs      : [101, 1996, 8870, 2024, 2652, 1012, 102]
attention_mask : [1, 1, 1, 1, 1, 1, 1]

special tokens (vi tri, token): [(0, '[CLS]'), (6, '[SEP]')]
so words                         : 4  ['The', 'cats', 'are', 'playing']
so tokens (gom special)          : 7
so tokens (khong gom special)    : 5  ['the', 'cats', 'are', 'playing', '.']
so subword tokens (bat dau '##') : 0  []

decode: [CLS] the cats are playing. [SEP]

[Nhan xet tu output]
- 4 words -> 5 content tokens (+2 special = 7).
- dau cau '.' tro thanh 1 token rieng.
- tu bi tach thanh >1 piece: khong co (moi word = 1 token)
- uncased: 'The' duoc ha chu thuong thanh 'the'


## 2. `tokens.keys()` — `input_ids` và `attention_mask`

- **`input_ids`** — tensor `(batch, seq_len)`, mỗi phần tử là **chỉ số trong vocab** của một token (đã thêm special token `[CLS]` đầu, `[SEP]` cuối). Đây là thứ duy nhất đi vào embedding lookup: `input_ids → embedding → Transformer`. Model không nhìn thấy chữ, chỉ thấy các ID này.
- **`attention_mask`** — tensor cùng shape, `1` = token hợp lệ, `0` = `[PAD]`. Dùng để self-attention **không cho padding tham gia như nội dung thật** (score tới vị trí `0` bị che trước softmax) và để pooling theo mask không tính padding.
- **`token_type_ids`** cũng có trong `keys()` với tokenizer này (toàn 0 vì chỉ có 1 câu; dùng để phân biệt câu A/B ở BERT). `DistilBertModel` **không nhận/không dùng** key này, nên pipeline của lab chỉ truyền `input_ids` + `attention_mask`.

In [3]:
print("token_type_ids:", tokens["token_type_ids"].tolist(), "(toan 0 vi chi co 1 cau; DistilBertModel khong dung)")
try:
    bert_tok = AutoTokenizer.from_pretrained("bert-base-uncased")
    print("bert-base-uncased keys:", list(bert_tok(text, return_tensors="pt").keys()))
    print("cung input_ids voi distilbert?", bert_tok(text)["input_ids"] == tok(text)["input_ids"])
except Exception as e:
    print("khong load duoc bert-base-uncased:", type(e).__name__)

token_type_ids: [[0, 0, 0, 0, 0, 0, 0]] (toan 0 vi chi co 1 cau; DistilBertModel khong dung)


bert-base-uncased keys: ['input_ids', 'token_type_ids', 'attention_mask']
cung input_ids voi distilbert? True


## 3. Batch có padding — thấy rõ `attention_mask = 0`

In [4]:
batch_texts = ["The cats are playing.", "This movie is great!", "I did not like the plot at all, although the acting was superb."]
b = tok(batch_texts, padding=True, truncation=True, return_tensors="pt")
print("input_ids shape     :", tuple(b["input_ids"].shape))
for t, ids_, m in zip(batch_texts, b["input_ids"], b["attention_mask"]):
    print("\ntext :", t)
    print("toks :", tok.convert_ids_to_tokens(ids_.tolist()))
    print("ids  :", ids_.tolist())
    print("mask :", m.tolist(), "-> so PAD =", int((m == 0).sum()))
print("\nPAD id =", tok.pad_token_id, "| vi tri mask==0 chinh la cac [PAD]:",
      bool(((b["input_ids"] == tok.pad_token_id) == (b["attention_mask"] == 0)).all()))

# vi du mask cua de bai (muc 7): [CLS] the cat eats fish [PAD] [PAD]
demo = tok("the cat eats fish", padding="max_length", max_length=7)
print("\nDemo de bai: ", tok.convert_ids_to_tokens(demo["input_ids"]), "\n  mask =", demo["attention_mask"])

input_ids shape     : (3, 17)

text : The cats are playing.
toks : ['[CLS]', 'the', 'cats', 'are', 'playing', '.', '[SEP]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]']
ids  : [101, 1996, 8870, 2024, 2652, 1012, 102, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
mask : [1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0] -> so PAD = 10

text : This movie is great!
toks : ['[CLS]', 'this', 'movie', 'is', 'great', '!', '[SEP]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]']
ids  : [101, 2023, 3185, 2003, 2307, 999, 102, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
mask : [1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0] -> so PAD = 10

text : I did not like the plot at all, although the acting was superb.
toks : ['[CLS]', 'i', 'did', 'not', 'like', 'the', 'plot', 'at', 'all', ',', 'although', 'the', 'acting', 'was', 'superb', '.', '[SEP]']
ids  : [101, 1045, 2106, 2025, 2066, 1996, 5436, 2012, 2035, 1010, 2348, 1996, 3772, 2001, 2

## 4. Từ hiếm / dài bị tách subword, và [UNK]

In [5]:
cands = ["unbelievable", "unbelievably", "disappointing", "cinematography", "antidisestablishmentarianism",
         "dehydrogenase", "COVID-19", "Schwarzenegger", "supercalifragilisticexpialidocious", "gooooood", "lol"]
rows = []
for w in cands:
    p = tok.tokenize(w)
    rows.append({"word": w, "n_tokens": len(p), "pieces": p})
df = pd.DataFrame(rows)
print(df.to_string(index=False))
whole = [r["word"] for r in rows if r["n_tokens"] == 1]
multi = [r for r in rows if r["n_tokens"] > 1]
print(f"\n[Nhan xet tu output] {len(multi)}/{len(rows)} tu bi tach; nhieu nhat: "
      f"{max(multi, key=lambda r: r['n_tokens'])['word']!r} -> {max(r['n_tokens'] for r in multi)} pieces; "
      f"nguyen khoi (1 token): {whole}")

                              word  n_tokens                                                                       pieces
                      unbelievable         1                                                               [unbelievable]
                      unbelievably         5                                               [un, ##bel, ##ie, ##va, ##bly]
                     disappointing         1                                                              [disappointing]
                    cinematography         1                                                             [cinematography]
      antidisestablishmentarianism         8                   [anti, ##dis, ##est, ##ab, ##lish, ##ment, ##arian, ##ism]
                     dehydrogenase         4                                                  [de, ##hy, ##dro, ##genase]
                          COVID-19         4                                                           [co, ##vid, -, 19]
                    Schw

In [6]:
# [UNK]: thu cac ky tu/chuoi lech khoi vocab WordPiece uncased
probes = ["I love it \U0001F600", "\u2764\ufe0f great", "\u4f60\u597d world", "caf\u00e9 \u2603", "\u0645\u0631\u062d\u0628\u0627", "\U0001F44D\U0001F44D"]
found = False
for s in probes:
    p = tok.tokenize(s)
    has_unk = tok.unk_token in p
    found |= has_unk
    print(f"{s!r:28} -> {p}   [UNK]: {has_unk}")
print("\nCo it nhat mot truong hop [UNK]:", found, "| unk id =", tok.unk_token_id)
print("Nhan xet tu output:", "co ky tu ngoai vocab -> bi thay bang [UNK], thong tin cua ky tu do mat hoan toan."
      if found else "khong tim thay [UNK] voi cac chuoi thu.")

'I love it 😀'                -> ['i', 'love', 'it', '[UNK]']   [UNK]: True
'❤️ great'                   -> ['[UNK]', 'great']   [UNK]: True
'你好 world'                   -> ['[UNK]', '[UNK]', 'world']   [UNK]: True
'café ☃'                     -> ['cafe', '[UNK]']   [UNK]: True
'مرحبا'                      -> ['م', '##ر', '##ح', '##ب', '##ا']   [UNK]: False
'👍👍'                         -> ['[UNK]']   [UNK]: True

Co it nhat mot truong hop [UNK]: True | unk id = 100
Nhan xet tu output: co ky tu ngoai vocab -> bi thay bang [UNK], thong tin cua ky tu do mat hoan toan.


## 5. Truncation trên dữ liệu thật (liên quan Exp 4: `max_len`)

In [7]:
import sys
sys.path.insert(0, ".")
import pretrained_classification as pc
A, B = pc.get_data("A"), pc.get_data("B")
def length_stats(df, n=3000):
    sub = df.sample(n, random_state=42)
    return np.array([len(x) for x in tok(sub["text"].tolist(), truncation=False)["input_ids"]])
LA, LB = length_stats(A["test"]), length_stats(B["test"])
rows = []
for name, L in (("A: IMDb test", LA), ("B: Yelp test", LB)):
    rows.append({"domain": name, "n": len(L), "mean": L.mean().round(1), "median": int(np.median(L)), "p90": int(np.percentile(L, 90)), "max": int(L.max()),
                 **{f"%>{T}": round(100 * (L > T).mean(), 1) for T in (64, 128, 256)}})
print(pd.DataFrame(rows).to_string(index=False))
print("\n(%>T = ty le review co so token (gom special) vuot T, tuc bi cat khi max_len=T; mau 3,000 review test moi domain, seed 42)")
print("\nVi du '<br />' trong IMDb tho:", tok.tokenize("great.<br /><br />movie")[:8], "-> clean_text bo di truoc khi tokenize:", repr(pc.clean_text("great.<br /><br />movie")))

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (679 > 512). Running this sequence through the model will result in indexing errors


      domain    n  mean  median  p90  max  %>64  %>128  %>256
A: IMDb test 3000 299.4     220  598 2166  97.9   87.9   40.9
B: Yelp test 3000 179.6     130  384 1433  78.6   50.4   20.9

(%>T = ty le review co so token (gom special) vuot T, tuc bi cat khi max_len=T; mau 3,000 review test moi domain, seed 42)

Vi du '<br />' trong IMDb tho: ['great', '.', '<', 'br', '/', '>', '<', 'br'] -> clean_text bo di truoc khi tokenize: 'great. movie'
